In [1]:
import os
import numpy as np
import pandas as pd
import torch
import os
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.nn import GCNConv

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

print("Imports completed")
from pathlib import Path

c:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\env\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Imports completed


In [2]:
PROJECT_ROOT = Path(
    r"C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation"
)


GRAPH_DIR = (
    PROJECT_ROOT /
    "final_data" /
    "graph"
)


TABULAR_DIR = (
    PROJECT_ROOT /
    "final_data" /
    "splits" /
    "tabular"
)


print(GRAPH_DIR)
print(TABULAR_DIR)

C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\final_data\graph
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\final_data\splits\tabular


In [3]:
# Load graph data

graph_data = torch.load(
    GRAPH_DIR / "graph_data_degree_only.pt",
    weights_only=False
)


# Load node mapping

with open(
    GRAPH_DIR / "node_to_id.pkl",
    "rb"
) as f:

    node_to_id = pickle.load(f)


# Load degree features

degree_features = np.load(
    GRAPH_DIR / "degree_features.npy"
)


print("="*90)
print("GRAPH ARTIFACTS")
print("="*90)


print(graph_data)


print(
    "Degree features:",
    degree_features.shape
)


print(
    "Nodes:",
    len(node_to_id)
)

GRAPH ARTIFACTS
Data(x=[13466, 3], edge_index=[2, 569486], y=[13466], train_mask=[13466], val_mask=[13466], test_mask=[13466])
Degree features: (13466, 3)
Nodes: 13466


In [4]:
# Load tabular splits

train_tabular = pd.read_csv(
    TABULAR_DIR / "train_tabular.csv"
)

val_tabular = pd.read_csv(
    TABULAR_DIR / "validation_tabular.csv"
)

test_tabular = pd.read_csv(
    TABULAR_DIR / "test_tabular.csv"
)


print("="*90)
print("TABULAR SPLITS")
print("="*90)


print(
    "Train:",
    train_tabular.shape
)

print(
    "Validation:",
    val_tabular.shape
)

print(
    "Test:",
    test_tabular.shape
)


print("\nColumns:")
print(
    train_tabular.columns.tolist()
)

TABULAR SPLITS
Train: (671, 43)
Validation: (144, 43)
Test: (144, 43)

Columns:
['screen_name', 'user_key', 'label_binary', 'followers_count', 'friends_count', 'favourites_count', 'listed_count', 'media_count', 'statuses_count', 'user_age', 'follower_growth_rate', 'friends_growth_rate', 'default_profile', 'default_profile_image', 'has_custom_timelines', 'possibly_sensitive', 'hashtag_in_description', 'numbers_in_description', 'no_type_tweet', 'no_type_retweet_with_comment', 'no_type_reply', 'mean_no_media_per_tweet', 'mean_no_words', 'no_languages', 'mean_no_hashtags', 'mean_favourites_per_tweet', 'time_between_tweets', 'tweet_frequency', 'min_tweets_per_hour', 'min_tweets_per_day', 'max_tweets_per_hour', 'max_tweets_per_day', 'max_occurence_of_same_gap', 'unique_mention_rate_per_tweet', 'mean_user_mentions_per_tweet', 'retweet_as_tweet_rate', 'no_retweet_tweets', 'mean_retweets_per_tweet', 'description_length', 'followers_friend_ratio', 'num_digits_in_name', 'num_digits_in_username', 

In [5]:
# Select tabular feature columns

tabular_feature_cols = [
    c for c in train_tabular.columns
    if c not in [
        "screen_name",
        "user_key",
        "label_binary"
    ]
]


print("="*90)
print("TABULAR FEATURE COLUMNS")
print("="*90)

print(
    "Number of features:",
    len(tabular_feature_cols)
)

print(
    tabular_feature_cols
)

TABULAR FEATURE COLUMNS
Number of features: 40
['followers_count', 'friends_count', 'favourites_count', 'listed_count', 'media_count', 'statuses_count', 'user_age', 'follower_growth_rate', 'friends_growth_rate', 'default_profile', 'default_profile_image', 'has_custom_timelines', 'possibly_sensitive', 'hashtag_in_description', 'numbers_in_description', 'no_type_tweet', 'no_type_retweet_with_comment', 'no_type_reply', 'mean_no_media_per_tweet', 'mean_no_words', 'no_languages', 'mean_no_hashtags', 'mean_favourites_per_tweet', 'time_between_tweets', 'tweet_frequency', 'min_tweets_per_hour', 'min_tweets_per_day', 'max_tweets_per_hour', 'max_tweets_per_day', 'max_occurence_of_same_gap', 'unique_mention_rate_per_tweet', 'mean_user_mentions_per_tweet', 'retweet_as_tweet_rate', 'no_retweet_tweets', 'mean_retweets_per_tweet', 'description_length', 'followers_friend_ratio', 'num_digits_in_name', 'num_digits_in_username', 'url_in_description']


In [7]:
# Combine all tabular splits

all_tabular = pd.concat(
    [
        train_tabular,
        val_tabular,
        test_tabular
    ],
    ignore_index=True
)


print("="*90)
print("ALL TABULAR DATA")
print("="*90)

print(
    "Total users:",
    len(all_tabular)
)

print(
    all_tabular["label_binary"].value_counts()
)

ALL TABULAR DATA
Total users: 959
label_binary
0    772
1    187
Name: count, dtype: int64


In [8]:
print("="*90)
print("LABEL DISTRIBUTION PER SPLIT")
print("="*90)


print("Train")
print(train_tabular["label_binary"].value_counts())


print("\nValidation")
print(val_tabular["label_binary"].value_counts())


print("\nTest")
print(test_tabular["label_binary"].value_counts())

LABEL DISTRIBUTION PER SPLIT
Train
label_binary
0    540
1    131
Name: count, dtype: int64

Validation
label_binary
0    116
1     28
Name: count, dtype: int64

Test
label_binary
0    116
1     28
Name: count, dtype: int64


In [9]:
print("="*90)
print("USER OVERLAP CHECK")
print("="*90)


train_users = set(train_tabular["user_key"])
val_users = set(val_tabular["user_key"])
test_users = set(test_tabular["user_key"])


print(
    "Train-Val overlap:",
    len(train_users & val_users)
)

print(
    "Train-Test overlap:",
    len(train_users & test_users)
)

print(
    "Val-Test overlap:",
    len(val_users & test_users)
)

USER OVERLAP CHECK
Train-Val overlap: 0
Train-Test overlap: 0
Val-Test overlap: 0


In [10]:
# Normalize keys

def normalize_key(x):
    return str(x).strip().lower()



all_tabular["user_key_norm"] = (
    all_tabular["user_key"]
    .apply(normalize_key)
)



tabular_feature_map = {}


for _, row in all_tabular.iterrows():

    key = row["user_key_norm"]

    features = (
        row[tabular_feature_cols]
        .values
        .astype(float)
    )

    tabular_feature_map[key] = features



print("="*90)
print("TABULAR FEATURE MAP")
print("="*90)


print(
    "Users in map:",
    len(tabular_feature_map)
)


sample_key = list(tabular_feature_map.keys())[0]


print(
    "Sample key:",
    sample_key
)


print(
    "Feature shape:",
    tabular_feature_map[sample_key].shape
)

TABULAR FEATURE MAP
Users in map: 959
Sample key: dokhi_sag
Feature shape: (40,)


In [11]:
# Build node features:
# Degree features (3) + Tabular features (40)

num_nodes = len(node_to_id)

num_tabular_features = len(tabular_feature_cols)


node_features = []

matched_tabular_nodes = 0


# Reverse mapping: id -> node name

id_to_node = {
    idx: name
    for name, idx in node_to_id.items()
}



for node_id in range(num_nodes):

    node_name = id_to_node[node_id]


    # Graph degree features

    degree_feat = degree_features[node_id]


    # Default tabular feature (for non-gold graph nodes)

    tab_feat = np.zeros(
        num_tabular_features,
        dtype=np.float32
    )


    node_key = normalize_key(node_name)


    if node_key in tabular_feature_map:

        tab_feat = tabular_feature_map[node_key]

        matched_tabular_nodes += 1



    combined_feature = np.concatenate(
        [
            degree_feat,
            tab_feat
        ]
    )


    node_features.append(
        combined_feature
    )



node_features = np.array(
    node_features,
    dtype=np.float32
)



print("="*90)
print("NODE FEATURES")
print("="*90)


print(
    "Shape:",
    node_features.shape
)


print(
    "Matched tabular nodes:",
    matched_tabular_nodes
)


print(
    "Missing tabular nodes:",
    num_nodes - matched_tabular_nodes
)

NODE FEATURES
Shape: (13466, 43)
Matched tabular nodes: 936
Missing tabular nodes: 12530


In [12]:
# Convert node features to tensor

x_tabular_graph = torch.tensor(
    node_features,
    dtype=torch.float32
)


print("="*90)
print("NEW NODE FEATURE TENSOR")
print("="*90)


print(
    "Shape:",
    x_tabular_graph.shape
)

NEW NODE FEATURE TENSOR
Shape: torch.Size([13466, 43])


In [13]:
# Replace node features with tabular + graph features

graph_tabular_data = graph_data.clone()


graph_tabular_data.x = x_tabular_graph



print("="*90)
print("GRAPH TABULAR DATA")
print("="*90)


print(graph_tabular_data)


print(
    "Node features:",
    graph_tabular_data.x.shape
)

GRAPH TABULAR DATA
Data(x=[13466, 43], edge_index=[2, 569486], y=[13466], train_mask=[13466], val_mask=[13466], test_mask=[13466])
Node features: torch.Size([13466, 43])


In [14]:
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import GCNConv



class GCNTabularClassifier(nn.Module):

    def __init__(self, input_dim=43):

        super().__init__()


        self.conv1 = GCNConv(
            input_dim,
            64
        )


        self.conv2 = GCNConv(
            64,
            32
        )


        self.classifier = nn.Linear(
            32,
            2
        )


    def forward(self, x, edge_index):

        x = self.conv1(
            x,
            edge_index
        )

        x = F.relu(x)


        x = self.conv2(
            x,
            edge_index
        )

        x = F.relu(x)


        x = self.classifier(
            x
        )


        return x



gcn_tabular_model = GCNTabularClassifier(
    input_dim=43
)


print("="*90)
print("GCN + TABULAR MODEL")
print("="*90)


print(gcn_tabular_model)

GCN + TABULAR MODEL
GCNTabularClassifier(
  (conv1): GCNConv(43, 64)
  (conv2): GCNConv(64, 32)
  (classifier): Linear(in_features=32, out_features=2, bias=True)
)


In [15]:
# Device

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)


# Move data and model

graph_tabular_data = graph_tabular_data.to(device)

gcn_tabular_model = gcn_tabular_model.to(device)



# Class weights

labels_train = (
    graph_tabular_data.y[
        graph_tabular_data.train_mask
    ]
)


class_counts = torch.bincount(
    labels_train
)


class_weights = (
    class_counts.sum()
    /
    (len(class_counts) * class_counts)
)



class_weights = class_weights.to(device)



print("="*90)
print("GCN + TABULAR CONFIGURATION")
print("="*90)


print("Device:")
print(device)


print("\nClass counts:")
print(class_counts)


print("\nClass weights:")
print(class_weights)

GCN + TABULAR CONFIGURATION
Device:
cpu

Class counts:
tensor([528, 129])

Class weights:
tensor([0.6222, 2.5465])


In [16]:
import torch.optim as optim


criterion = nn.CrossEntropyLoss(
    weight=class_weights
)


optimizer = optim.AdamW(
    gcn_tabular_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)



print("="*90)
print("TRAINING CONFIGURATION")
print("="*90)


print(
    "Optimizer: AdamW"
)

print(
    "Learning rate: 1e-3"
)

print(
    "Loss: CrossEntropyLoss"
)

print(
    "Weight decay: 1e-4"
)

TRAINING CONFIGURATION
Optimizer: AdamW
Learning rate: 1e-3
Loss: CrossEntropyLoss
Weight decay: 1e-4


In [17]:
import copy


epochs = 200


best_val_loss = float("inf")

best_model_state = None



print("="*90)
print("TRAINING GCN + TABULAR")
print("="*90)



for epoch in range(1, epochs + 1):

    gcn_tabular_model.train()


    optimizer.zero_grad()


    out = gcn_tabular_model(
        graph_tabular_data.x,
        graph_tabular_data.edge_index
    )


    loss = criterion(
        out[graph_tabular_data.train_mask],
        graph_tabular_data.y[graph_tabular_data.train_mask]
    )


    loss.backward()

    optimizer.step()



    # Validation

    gcn_tabular_model.eval()


    with torch.no_grad():

        val_out = gcn_tabular_model(
            graph_tabular_data.x,
            graph_tabular_data.edge_index
        )


        val_loss = criterion(
            val_out[graph_tabular_data.val_mask],
            graph_tabular_data.y[graph_tabular_data.val_mask]
        )



    if val_loss.item() < best_val_loss:

        best_val_loss = val_loss.item()

        best_model_state = copy.deepcopy(
            gcn_tabular_model.state_dict()
        )



    if epoch % 20 == 0:

        print(
            f"Epoch [{epoch}/{epochs}] "
            f"Train Loss: {loss.item():.4f} "
            f"Val Loss: {val_loss.item():.4f}"
        )



print("\nTraining completed")



# Restore best model

gcn_tabular_model.load_state_dict(
    best_model_state
)


print("Best GCN + Tabular model restored")

TRAINING GCN + TABULAR
Epoch [20/200] Train Loss: 1.1000 Val Loss: 0.8479
Epoch [40/200] Train Loss: 0.7010 Val Loss: 0.6854
Epoch [60/200] Train Loss: 0.6786 Val Loss: 0.6807
Epoch [80/200] Train Loss: 0.6734 Val Loss: 0.6790
Epoch [100/200] Train Loss: 0.6679 Val Loss: 0.6748
Epoch [120/200] Train Loss: 0.6622 Val Loss: 0.6712
Epoch [140/200] Train Loss: 0.6553 Val Loss: 0.6647
Epoch [160/200] Train Loss: 0.6487 Val Loss: 0.6571
Epoch [180/200] Train Loss: 0.6390 Val Loss: 0.6507
Epoch [200/200] Train Loss: 0.6311 Val Loss: 0.6440

Training completed
Best GCN + Tabular model restored


In [18]:
# Prediction for GCN + Tabular

gcn_tabular_model.eval()


with torch.no_grad():

    logits = gcn_tabular_model(
        graph_tabular_data.x,
        graph_tabular_data.edge_index
    )


    probabilities = torch.softmax(
        logits,
        dim=1
    )[:,1]



val_prob_gcn_tabular = (
    probabilities[
        graph_tabular_data.val_mask
    ]
    .cpu()
    .numpy()
)


test_prob_gcn_tabular = (
    probabilities[
        graph_tabular_data.test_mask
    ]
    .cpu()
    .numpy()
)



print("="*90)
print("GCN + TABULAR PREDICTION")
print("="*90)


print(
    "Validation:",
    val_prob_gcn_tabular.shape
)


print(
    "Test:",
    test_prob_gcn_tabular.shape
)

GCN + TABULAR PREDICTION
Validation: (138,)
Test: (140,)


In [19]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)


def evaluate_gcn_model(
    probabilities,
    mask_name,
    graph_data
):

    if mask_name == "Validation":
        mask = graph_data.val_mask

    else:
        mask = graph_data.test_mask


    y_true = (
        graph_data.y[mask]
        .cpu()
        .numpy()
    )


    y_prob = probabilities


    y_pred = (
        y_prob >= 0.5
    ).astype(int)



    print("="*90)
    print(mask_name)
    print("="*90)


    print(
        "Accuracy: ",
        round(
            accuracy_score(
                y_true,
                y_pred
            ),
            4
        )
    )


    print(
        "Precision:",
        round(
            precision_score(
                y_true,
                y_pred,
                zero_division=0
            ),
            4
        )
    )


    print(
        "Recall:   ",
        round(
            recall_score(
                y_true,
                y_pred,
                zero_division=0
            ),
            4
        )
    )


    print(
        "F1:       ",
        round(
            f1_score(
                y_true,
                y_pred,
                zero_division=0
            ),
            4
        )
    )


    print(
        "ROC_AUC:  ",
        round(
            roc_auc_score(
                y_true,
                y_prob
            ),
            4
        )
    )


    print(
        "PR_AUC:   ",
        round(
            average_precision_score(
                y_true,
                y_prob
            ),
            4
        )
    )


    print("\nConfusion Matrix:")
    print(
        confusion_matrix(
            y_true,
            y_pred
        )
    )


    print("\nClassification Report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=[
                "Human",
                "Bot"
            ],
            zero_division=0
        )
    )



evaluate_gcn_model(
    val_prob_gcn_tabular,
    "Validation",
    graph_tabular_data
)


evaluate_gcn_model(
    test_prob_gcn_tabular,
    "Test",
    graph_tabular_data
)

Validation
Accuracy:  0.6304
Precision: 0.3235
Recall:    0.8148
F1:        0.4632
ROC_AUC:   0.705
PR_AUC:    0.3368

Confusion Matrix:
[[65 46]
 [ 5 22]]

Classification Report:
              precision    recall  f1-score   support

       Human       0.93      0.59      0.72       111
         Bot       0.32      0.81      0.46        27

    accuracy                           0.63       138
   macro avg       0.63      0.70      0.59       138
weighted avg       0.81      0.63      0.67       138

Test
Accuracy:  0.6429
Precision: 0.3115
Recall:    0.7037
F1:        0.4318
ROC_AUC:   0.7089
PR_AUC:    0.3461

Confusion Matrix:
[[71 42]
 [ 8 19]]

Classification Report:
              precision    recall  f1-score   support

       Human       0.90      0.63      0.74       113
         Bot       0.31      0.70      0.43        27

    accuracy                           0.64       140
   macro avg       0.61      0.67      0.59       140
weighted avg       0.79      0.64      0.68   

In [20]:
import pandas as pd
import os


results_dir = Path(
    "../results"
)


results_dir.mkdir(
    exist_ok=True
)



gcn_tabular_results = pd.DataFrame(
    [
        {
            "Model": "GCN-Tabular Validation",
            "Accuracy": 0.6304,
            "Precision": 0.3235,
            "Recall": 0.8148,
            "F1": 0.4632,
            "ROC_AUC": 0.7050,
            "PR_AUC": 0.3368
        },

        {
            "Model": "GCN-Tabular Test",
            "Accuracy": 0.6429,
            "Precision": 0.3115,
            "Recall": 0.7037,
            "F1": 0.4318,
            "ROC_AUC": 0.7089,
            "PR_AUC": 0.3461
        }
    ]
)



gcn_tabular_results.to_csv(
    results_dir / "gcn_tabular_results.csv",
    index=False
)



print("="*90)
print("GCN TABULAR RESULTS SAVED")
print("="*90)


print(
    results_dir / "gcn_tabular_results.csv"
)


gcn_tabular_results

GCN TABULAR RESULTS SAVED
..\results\gcn_tabular_results.csv


,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
0,GCN-Tabular Validation,0.6304,0.3235,0.8148,0.4632,0.7050,0.3368
1,GCN-Tabular Test,0.6429,0.3115,0.7037,0.4318,0.7089,0.3461
